# A/B Testing XP


In [ ]:
# Run once in Google Colab
!pip -q install statsmodels

import math
from statistics import NormalDist

import numpy as np
from statsmodels.stats.power import TTestIndPower

ALPHA = 0.05
DEFAULT_POWER = 0.80
NORMAL = NormalDist()

def required_per_group(effect_size, power=DEFAULT_POWER):
    """Rounded-up equal-group sample size for a two-sided t test."""
    sample_size = TTestIndPower().solve_power(
        effect_size=effect_size, alpha=ALPHA, power=power,
        ratio=1.0, alternative='two-sided'
    )
    return math.ceil(sample_size)

## Exercise 1 — Calculating Required Sample Size

With a two-sided significance level of 0.05, 80% power, equal group sizes, and the supplied standardized effect size of 0.30, the required sample is **175 users per group** (350 total after rounding up).

`TTestIndPower` uses a standardized effect size (Cohen's *d*). A change from 20% to 23% is a proportion comparison; in a real email experiment, use a two-proportion calculation based on the actual baseline rate and minimum detectable lift.

In [ ]:
exercise_1_n = required_per_group(0.30, power=0.80)
print(f'Required sample size per group: {exercise_1_n:,}')
print(f'Total sample size: {exercise_1_n * 2:,}')

## Exercise 2 — Relationship Between Effect Size and Sample Size

A larger effect is easier to distinguish from normal sampling noise, so it requires fewer observations. The relationship is approximately inverse-square: halving the effect size requires roughly four times as many observations.

In [ ]:
print('Effect size | Required sample per group')
print('------------|--------------------------')
for effect_size in (0.20, 0.40, 0.50):
    print(f'{effect_size:>11.2f} | {required_per_group(effect_size):>25,}')

## Exercise 3 — Impact of Statistical Power

Increasing power reduces the chance of missing a real effect (a false negative), but it needs more traffic and time. Selecting power before the test makes this trade-off explicit and prevents an underpowered experiment.

In [ ]:
print('Power | Required sample per group (effect size = 0.20)')
print('------|--------------------------------------------------')
for power in (0.70, 0.80, 0.90):
    print(f'{power:>5.2f} | {required_per_group(0.20, power):>49,}')

## Exercise 4 — Sequential Testing

Pre-register a maximum sample size, weekly interim looks, a practical minimum lift, data-quality checks, a harm guardrail, and a futility rule. Do not use `p < 0.05` every week: repeated peeking inflates the false-positive rate.

The code below illustrates an approximate O'Brien–Fleming alpha-spending rule for four equally sized weekly looks. With a preplanned schedule, week-three `p = 0.02` crosses the approximate efficacy boundary, so stop only after confirming B has the expected direction, meets the practical-lift requirement, and passes guardrails. If the sequential plan was not set in advance, do not stop using the unadjusted p-value; continue to the planned end or apply an appropriate correction.

In [ ]:
def obrien_fleming_boundary(information_fraction):
    """Approximate two-sided O'Brien-Fleming z and p boundary."""
    if not 0 < information_fraction <= 1:
        raise ValueError('information_fraction must be between 0 and 1')
    fixed_sample_z = NORMAL.inv_cdf(1 - ALPHA / 2)
    z_boundary = fixed_sample_z / math.sqrt(information_fraction)
    return z_boundary, 2 * (1 - NORMAL.cdf(z_boundary))

print('Week | Stop for efficacy if two-sided p <=')
for week in range(1, 5):
    _, p_boundary = obrien_fleming_boundary(week / 4)
    print(f'  {week}  | {p_boundary:.4f}')

_, week_three_boundary = obrien_fleming_boundary(0.75)
print(f'\nWeek 3: p = 0.020 and boundary = {week_three_boundary:.4f}.')

## Exercise 5 — Bayesian A/B Testing

For a binary engagement metric, assign both versions a neutral `Beta(1, 1)` prior. It is symmetric, so it represents the initial 50/50 belief about which version is better. After observing data, update each engagement rate to `Beta(1 + successes, 1 + failures)`.

Estimate `P(B > A)` by drawing from the two posterior distributions. A 65% probability that B is better is encouraging, but normally below a pre-agreed 90–95% launch threshold; collect more data or retain the control. At 55%, evidence is weak: continue only if more data is worthwhile, otherwise stop for futility.

In [ ]:
def probability_b_beats_a(a_successes, a_trials, b_successes, b_trials, draws=200_000, seed=42):
    """Estimate P(B engagement rate > A engagement rate)."""
    if not 0 <= a_successes <= a_trials or not 0 <= b_successes <= b_trials:
        raise ValueError('successes must be between zero and the number of trials')
    rng = np.random.default_rng(seed)
    a_rate = rng.beta(1 + a_successes, 1 + a_trials - a_successes, draws)
    b_rate = rng.beta(1 + b_successes, 1 + b_trials - b_successes, draws)
    return float(np.mean(b_rate > a_rate))

# Example: replace these counts with your experiment's results.
posterior_probability = probability_b_beats_a(200, 1_000, 225, 1_000)
print(f'Estimated P(B > A): {posterior_probability:.1%}')

## Exercise 6 — Adaptive Experimentation

Start layouts A, B, and C at 33% each. After week 1, consider uncertainty as well as raw engagement, then shift some traffic to C using posterior probability of being best (for example, Thompson sampling). Keep an exploration floor such as 15% for every layout, recompute after each planned batch, cap abrupt changes, and retain allocation logs.

Challenges include noisy early winners, changing traffic mix, unfair exposure, harder inference, and operational complexity. Address these with exploration floors, randomized assignment, batch updates, logged allocation probabilities, segment and novelty checks, simulations before launch, and adaptive-trial analysis.

In [ ]:
def thompson_allocation(alpha_params, beta_params, draws=100_000):
    """Traffic shares based on each layout's probability of being best."""
    if len(alpha_params) != len(beta_params):
        raise ValueError('alpha_params and beta_params must have equal length')
    rng = np.random.default_rng(7)
    sampled_rates = rng.beta(alpha_params, beta_params, size=(draws, len(alpha_params)))
    winners = np.argmax(sampled_rates, axis=1)
    return np.bincount(winners, minlength=len(alpha_params)) / draws

# Example after week 1: A=200/1000, B=205/1000, C=225/1000 engagements.
shares = thompson_allocation(
    alpha_params=np.array([201, 206, 226]),
    beta_params=np.array([801, 796, 776]),
)
for layout, share in zip(('A', 'B', 'C'), shares):
    print(f'Layout {layout}: {share:.1%} suggested traffic before applying exploration floors')